# Model Distillation for Computer Vision
## Part 2: Auto-labeling with Foundation Models

**Workshop Duration:** 2 hours  
**Level:** Intermediate

---

### Learning Objectives
By the end of this notebook, you will:
1. Use Grounding DINO for text-prompted object detection
2. Design effective ontologies for auto-labeling
3. Generate a training dataset from unlabeled images

---

## 1. Understanding Grounding DINO

**Grounding DINO** is an open-vocabulary object detector that takes:
- An image
- A text prompt describing what to find

And outputs bounding boxes for all matching objects.

```
┌─────────────┐     ┌──────────────────┐     ┌─────────────────┐
│   Image     │ → │  Grounding DINO  │ → │ Bounding Boxes  │
│             │     │  + Text Prompt   │     │ + Confidence    │
└─────────────┘     └──────────────────┘     └─────────────────┘
```

### Key Stats
- **52.5 AP** on COCO zero-shot (no training on COCO data)
- **Open vocabulary:** detect any object describable in text
- **GPU Memory:** 8-16GB depending on variant

In [ ]:
# Setup imports
import torch
import requests
from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import os

# Check GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 2. Demo: Text-Prompted Detection with Grounding DINO

Let's see how Grounding DINO detects objects using natural language prompts.

In [ ]:
# Load Grounding DINO from Hugging Face Transformers
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

model_id = "IDEA-Research/grounding-dino-tiny"
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForZeroShotObjectDetection.from_pretrained(model_id).to(device)

print(f"Loaded: {model_id}")

In [ ]:
# Load a sample image
image_url = "http://images.cocodataset.org/val2017/000000039769.jpg"
response = requests.get(image_url)
image = Image.open(BytesIO(response.content))

# Display the image
plt.figure(figsize=(10, 8))
plt.imshow(image)
plt.axis('off')
plt.title('Input Image')
plt.show()

In [ ]:
def detect_with_text(image, text_labels, box_threshold=0.35, text_threshold=0.25):
    """
    Run Grounding DINO detection with text prompts.
    
    Args:
        image: PIL Image
        text_labels: List of text descriptions to detect
        box_threshold: Confidence threshold for boxes
        text_threshold: Confidence threshold for text matching
    
    Returns:
        Dictionary with boxes, scores, and labels
    """
    # Prepare inputs
    inputs = processor(
        images=image,
        text=[text_labels],  # Needs to be nested list
        return_tensors="pt"
    ).to(device)
    
    # Run inference
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Post-process results
    results = processor.post_process_grounded_object_detection(
        outputs,
        inputs.input_ids,
        threshold=box_threshold,
        text_threshold=text_threshold,
        target_sizes=[image.size[::-1]]
    )
    
    return results[0]

In [ ]:
# Detect cats and remotes
text_labels = ["a cat", "a remote control"]
results = detect_with_text(image, text_labels)

print(f"Detected {len(results['boxes'])} objects:")
for box, score, label in zip(results['boxes'], results['scores'], results['labels']):
    print(f"  - {label}: {score:.3f}")

In [ ]:
def visualize_detections(image, results, title="Detections"):
    """
    Visualize detection results on the image.
    """
    fig, ax = plt.subplots(1, figsize=(12, 8))
    ax.imshow(image)
    
    colors = plt.cm.rainbow(torch.linspace(0, 1, len(set(results['labels']))))
    label_colors = {label: colors[i] for i, label in enumerate(set(results['labels']))}
    
    for box, score, label in zip(results['boxes'], results['scores'], results['labels']):
        x0, y0, x1, y1 = box.tolist()
        width = x1 - x0
        height = y1 - y0
        
        color = label_colors[label]
        rect = patches.Rectangle(
            (x0, y0), width, height,
            linewidth=2, edgecolor=color, facecolor='none'
        )
        ax.add_patch(rect)
        
        ax.text(
            x0, y0 - 5,
            f"{label}: {score:.2f}",
            fontsize=10, color='white',
            bbox=dict(boxstyle='round', facecolor=color, alpha=0.8)
        )
    
    ax.axis('off')
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

visualize_detections(image, results, "Grounding DINO Detection")

---

## 3. Exercise: Design Your Own Ontology

An **ontology** in Autodistill maps:
- **Text prompts** (what the base model looks for)
- **Class names** (what gets saved in annotations)

```python
ontology = CaptionOntology({
    "text prompt": "class_name",
    "another prompt": "another_class"
})
```

### Task: Try Different Prompts

Experiment with different text prompts to see how Grounding DINO responds.

In [ ]:
# EXERCISE: Try different prompts
# Modify the text_labels list below and run the cell

text_labels = [
    # TODO: Add your own prompts here
    "a cat",
    "a couch",
    "a blanket",
]

results = detect_with_text(image, text_labels, box_threshold=0.3)
visualize_detections(image, results, f"Custom Detection: {text_labels}")

### Prompt Engineering Tips

| Technique | Example | Why It Works |
|-----------|---------|-------------|
| **Add articles** | "a cat" vs "cat" | More natural language |
| **Use colors** | "white cat" | Reduces false positives |
| **Add context** | "cat on couch" | Disambiguates similar objects |
| **Be specific** | "tabby cat" | Targets specific variants |

In [ ]:
# Compare prompt variations
prompt_variations = [
    ["cat"],
    ["a cat"],
    ["orange cat"],
    ["cat sitting on furniture"]
]

print("Prompt Comparison Results:")
print("=" * 50)
for prompts in prompt_variations:
    results = detect_with_text(image, prompts, box_threshold=0.25)
    detections = len(results['boxes'])
    avg_score = sum(results['scores']).item() / detections if detections > 0 else 0
    print(f"Prompt: {prompts[0]:30} | Detections: {detections} | Avg score: {avg_score:.3f}")

---

## 4. Auto-labeling with Autodistill

Now let's use the full Autodistill workflow to auto-label a batch of images.

In [ ]:
# Import Autodistill components
from autodistill.detection import CaptionOntology
from autodistill_grounding_dino import GroundingDINO

# Define our ontology for the workshop
# This maps what we're looking for to how we'll label it
ontology = CaptionOntology({
    "cat": "cat",
    "remote control": "remote",
    "couch": "couch"
})

print("Ontology:")
for prompt, cls in zip(ontology.prompts(), ontology.classes()):
    print(f"  '{prompt}' → '{cls}'")

In [ ]:
# Initialize the base model with our ontology
base_model = GroundingDINO(ontology=ontology)
print("Base model initialized!")

### Create Sample Dataset for Labeling

Let's download a few images to demonstrate auto-labeling.

In [ ]:
# Create directories
os.makedirs("../data/unlabeled_images", exist_ok=True)
os.makedirs("../data/labeled_dataset", exist_ok=True)

# Download sample images from COCO
sample_urls = [
    "http://images.cocodataset.org/val2017/000000039769.jpg",  # Cats
    "http://images.cocodataset.org/val2017/000000037777.jpg",  # Living room
    "http://images.cocodataset.org/val2017/000000087038.jpg",  # Cat
    "http://images.cocodataset.org/val2017/000000057597.jpg",  # Couch
    "http://images.cocodataset.org/val2017/000000025560.jpg",  # Living room
]

print("Downloading sample images...")
for i, url in enumerate(sample_urls):
    response = requests.get(url)
    img = Image.open(BytesIO(response.content))
    img.save(f"../data/unlabeled_images/image_{i+1}.jpg")
    print(f"  Downloaded image_{i+1}.jpg")

print(f"\nSaved {len(sample_urls)} images to ../data/unlabeled_images/")

In [ ]:
# Display the downloaded images
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, ax in enumerate(axes):
    img = Image.open(f"../data/unlabeled_images/image_{i+1}.jpg")
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f"image_{i+1}.jpg")
plt.tight_layout()
plt.show()

### Run Auto-Labeling

In [ ]:
# Auto-label all images using Grounding DINO
print("Starting auto-labeling...")
print("This may take 1-2 minutes depending on GPU.")

base_model.label(
    input_folder="../data/unlabeled_images",
    output_folder="../data/labeled_dataset",
    extension=".jpg"
)

print("\nAuto-labeling complete!")

In [ ]:
# Examine the generated dataset structure
print("Generated dataset structure:")
for root, dirs, files in os.walk("../data/labeled_dataset"):
    level = root.replace("../data/labeled_dataset", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = "  " * (level + 1)
    for file in files[:5]:  # Show first 5 files
        print(f"{subindent}{file}")
    if len(files) > 5:
        print(f"{subindent}... and {len(files) - 5} more files")

In [ ]:
# Read and display the data.yaml configuration
yaml_path = "../data/labeled_dataset/data.yaml"
if os.path.exists(yaml_path):
    with open(yaml_path, 'r') as f:
        print("data.yaml contents:")
        print("=" * 40)
        print(f.read())

In [ ]:
# Visualize some auto-labeled images
import supervision as sv

# Find a label file and corresponding image
label_dir = "../data/labeled_dataset/train/labels"
image_dir = "../data/labeled_dataset/train/images"

if os.path.exists(label_dir):
    label_files = [f for f in os.listdir(label_dir) if f.endswith('.txt')]
    
    if label_files:
        # Read first label file
        label_file = label_files[0]
        image_file = label_file.replace('.txt', '.jpg')
        
        print(f"Label file: {label_file}")
        with open(os.path.join(label_dir, label_file), 'r') as f:
            print("Contents (YOLO format):")
            print(f.read())
        print("\nFormat: class_id x_center y_center width height (all normalized 0-1)")

---

## 5. Exercise: Create Your Own Ontology

Now it's your turn! Create an ontology for a different detection task.

In [ ]:
# EXERCISE: Define your own ontology
# Suggestions:
# - Safety equipment: hard hat, safety vest, goggles
# - Retail: bottle, can, box
# - Traffic: car, truck, pedestrian, bicycle

my_ontology = CaptionOntology({
    # TODO: Fill in your ontology
    # "text prompt": "class_name",
    "person": "person",
    "television": "tv",
    "book": "book",
})

print("Your ontology:")
for prompt, cls in zip(my_ontology.prompts(), my_ontology.classes()):
    print(f"  '{prompt}' → '{cls}'")

In [ ]:
# Test your ontology on a sample image
my_base_model = GroundingDINO(ontology=my_ontology)

# Run prediction on one image
test_image_path = "../data/unlabeled_images/image_1.jpg"
detections = my_base_model.predict(test_image_path)

print(f"Detected {len(detections.xyxy)} objects with your ontology")

---

## 6. Label Quality Assessment

Before training, always verify the quality of auto-generated labels.

In [ ]:
# Count annotations per class
class_counts = {}

label_dir = "../data/labeled_dataset/train/labels"
if os.path.exists(label_dir):
    for label_file in os.listdir(label_dir):
        if label_file.endswith('.txt'):
            with open(os.path.join(label_dir, label_file), 'r') as f:
                for line in f:
                    if line.strip():
                        class_id = int(line.split()[0])
                        class_counts[class_id] = class_counts.get(class_id, 0) + 1

print("Annotation counts by class:")
print("=" * 40)
class_names = list(ontology.classes())
for class_id, count in sorted(class_counts.items()):
    class_name = class_names[class_id] if class_id < len(class_names) else f"class_{class_id}"
    print(f"  {class_name}: {count} annotations")

total = sum(class_counts.values())
print(f"\nTotal annotations: {total}")

### Quality Checklist

Before proceeding to training, verify:

- [ ] Each class has at least 100 annotations (ideally 500+)
- [ ] No class is severely underrepresented
- [ ] Spot-check 10-20 images for correct labeling
- [ ] Bounding boxes are reasonably tight around objects

---

## 7. Summary

### What We Learned

1. **Grounding DINO** detects objects using natural language text prompts
2. **Ontologies** map text prompts to class names for annotations
3. **Prompt engineering** improves detection quality (use articles, colors, context)
4. **Autodistill** automates the labeling → dataset creation workflow
5. **Quality assessment** is critical before training

### Dataset Output

We've created:
- `../data/labeled_dataset/train/images/` - Training images
- `../data/labeled_dataset/train/labels/` - YOLO format annotations
- `../data/labeled_dataset/valid/` - Validation split
- `../data/labeled_dataset/data.yaml` - Dataset configuration

### Next Steps

In **Notebook 3**, we'll:
- Train a YOLO26 model on our auto-labeled dataset
- Evaluate model performance
- Export for edge deployment

---

**Continue to:** [03_training_yolo26.ipynb](./03_training_yolo26.ipynb)